In [1]:
import pandas as pd
import numpy as np

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
DATA_DIR = "/content/drive/MyDrive/morocco-purchasing-power/"

## Loading data

In [3]:
monthly_df = pd.read_csv(DATA_DIR + "hcp_monthly_category.csv")
yearly_df = pd.read_csv(DATA_DIR + "hcp_yearly_category.csv")
city_df = pd.read_csv(DATA_DIR + "hcp_city_indices.csv")

print(f"monthly: {monthly_df.shape}")
print(f"yearly: {yearly_df.shape}")
print(f"city: {city_df.shape}")

monthly: (2706, 6)
yearly: (2636, 9)
city: (3320, 6)


In [4]:
monthly_df.head()

,category,index_prev_period,index_curr_period,change_pct,month,year
0,Produits alimentaires,128.0,125.6,-1.9,Juillet,2026
1,01 - Produits alimentaires et boissons non alc...,127.2,124.6,-2.0,Juillet,2026
2,02 - Boissons alcoolisées et tabac,150.5,150.5,0.0,Juillet,2026
3,Produits non alimentaires,114.6,114.3,-0.3,Juillet,2026
4,03 - Articles d'habillements et chaussures,117.9,117.7,-0.2,Juillet,2026


In [5]:
monthly_df.dtypes

,0
category,object
index_prev_period,float64
index_curr_period,float64
change_pct,float64
month,object
year,int64


In [6]:
monthly_df.isna().sum()

,0
category,42
index_prev_period,37
index_curr_period,37
change_pct,39
month,0
year,0


## Dropping concatenated-garbage rows (all categories/cities joined into one cell)

In [7]:
garbage_mask_m = monthly_df["category"].astype(str).str.contains(r"\n", regex=True, na=False)
print(f"monthly: dropping {garbage_mask_m.sum()} concatenated-garbage rows")

monthly: dropping 13 concatenated-garbage rows


In [8]:
monthly_df = monthly_df[~garbage_mask_m].copy()

In [9]:
garbage_mask_y = yearly_df["category"].astype(str).str.contains(r"\n", regex=True, na=False)
print(f"yearly: dropping {garbage_mask_y.sum()} concatenated-garbage rows")

yearly: dropping 14 concatenated-garbage rows


In [10]:
yearly_df = yearly_df[~garbage_mask_y].copy()

In [11]:
garbage_mask_c = city_df["city"].astype(str).str.contains(r"\n", regex=True, na=False)
print(f"city: dropping {garbage_mask_c.sum()} concatenated-garbage rows")
city_df = city_df[~garbage_mask_c].copy()

city: dropping 11 concatenated-garbage rows


## Dropping junk/header-leak rows (NaN category/city, and remaining numbers are either empty or look like leftover year labels):

In [12]:
value_cols_m = ["index_prev_period","index_curr_period","change_pct"]
value_cols_y = ["index_ref_period","index_curr_period","change_pct","cum_index_prior","cum_index_curr","cum_change_pct"]
value_cols_c = ["index_prev_period","index_curr_period","change_pct"]

In [13]:
# monthly
nan_cat_m = monthly_df["category"].isna()
vals_m = monthly_df.loc[nan_cat_m, value_cols_m]

In [14]:
print(f"monthly: {nan_cat_m.sum()} NaN categories")

monthly: 42 NaN categories


In [15]:
looks_like_years_m = vals_m.apply(lambda row: all((pd.isna(v)) or (1990 <= v <= 2030) for v in row), axis=1)
junk_mask_m = nan_cat_m & looks_like_years_m
print(f"monthly: dropping {junk_mask_m.sum()} junk/header-leak rows")

monthly: dropping 27 junk/header-leak rows


In [16]:
monthly_df = monthly_df[~junk_mask_m].copy()

In [17]:
# yearly
nan_cat_y = yearly_df["category"].isna()
vals_y = yearly_df.loc[nan_cat_y, value_cols_y]
looks_like_years_y = vals_y.apply(lambda row: all((pd.isna(v)) or (1990 <= v <= 2030) for v in row), axis=1)
junk_mask_y = nan_cat_y & looks_like_years_y
print(f"yearly: dropping {junk_mask_y.sum()} junk/header-leak rows")
yearly_df = yearly_df[~junk_mask_y].copy()

yearly: dropping 27 junk/header-leak rows


In [18]:
# city
nan_cat_c = city_df["city"].isna()
vals_c = city_df.loc[nan_cat_c, value_cols_c]
looks_like_years_c = vals_c.apply(lambda row: all((pd.isna(v)) or (1990 <= v <= 2030) for v in row), axis=1)
junk_mask_c = nan_cat_c & looks_like_years_c
print(f"city: dropping {junk_mask_c.sum()} junk/header-leak rows")
city_df = city_df[~junk_mask_c].copy()

city: dropping 27 junk/header-leak rows


## Recovering Mars 2010 positionally

In [19]:
reference = monthly_df[(monthly_df["year"]==2010) & (monthly_df["month"]=="février")]
category_template = reference["category"].tolist()
category_template

['Produits alimentaires',
 '01 - Produits alimentaires et boissons non alcoolisées',
 '02 - Boissons alcoolisées et tabac',
 'Produits non alimentaires',
 "03 - Articles d'habillements et chaussures",
 '04 - Logements, eau, électricité et autres combustibles',
 '05 - Meubles, articles de ménages et entretien courant du foyer',
 '06 - Santé',
 '07 - Transport',
 '08 - Communication',
 '09 - Loisirs et culture',
 '10 - Enseignement',
 '11 - Restaurants et hôtels',
 '12 - Biens et services divers',
 'Ensemble']

In [20]:
mars_2010_idx = monthly_df[(monthly_df["year"]==2010) & (monthly_df["month"]=="mars")].index
print(f"Mars 2010 rows: {len(mars_2010_idx)}, template length: {len(category_template)}")

Mars 2010 rows: 15, template length: 15


In [21]:
if len(mars_2010_idx) == len(category_template):
    monthly_df.loc[mars_2010_idx, "category"] = category_template

## Dropping any remaining unrecovered NaN rows

In [22]:
monthly_df = monthly_df[monthly_df["category"].notna()].copy()
yearly_df = yearly_df[yearly_df["category"].notna()].copy()
city_df = city_df[city_df["city"].notna()].copy()

print("monthly NaN:", monthly_df.isna().sum().sum())
print("yearly NaN:", yearly_df.isna().sum().sum())
print("city NaN:", city_df.isna().sum().sum())

monthly NaN: 0
yearly NaN: 0
city NaN: 0


## Normalizing category and city names

In [23]:
import re

categories = {
    "01": "01 - Produits alimentaires et boissons non alcoolisées", "02": "02 - Boissons alcoolisées et tabac",
    "03": "03 - Articles d'habillement et chaussures", "04": "04 - Logement, eau, électricité et autres combustibles",
    "05": "05 - Meubles, articles de ménage et entretien courant du foyer", "06": "06 - Santé",
    "07": "07 - Transport", "08": "08 - Communication", "09": "09 - Loisirs et culture",
    "10": "10 - Enseignement", "11": "11 - Restaurants et hôtels", "12": "12 - Biens et services divers",
}


In [24]:
import re
pattern = re.compile(r"^(\d{2})\s*[-–]")

monthly_df["category"] = monthly_df["category"].apply(
    lambda cat: categories[pattern.match(cat).group(1)] if pattern.match(cat) else cat
)
yearly_df["category"] = yearly_df["category"].apply(
    lambda cat: categories[pattern.match(cat).group(1)] if pattern.match(cat) else cat
)

In [25]:
print("monthly categories:", monthly_df["category"].nunique())
print("yearly categories:", yearly_df["category"].nunique())

monthly categories: 15
yearly categories: 15


In [26]:
city_name_map = {"Al-hoceima": "Al-Hoceima", "Beni-Mellal": "Béni-Mellal", "Edakhla": "Dakhla"}
city_df["city"] = city_df["city"].replace(city_name_map)
print(sorted(city_df["city"].unique()))

['Agadir', 'Al-Hoceima', 'Béni-Mellal', 'Casablanca', 'Dakhla', 'Ensemble', 'Errachidia', 'Fès', 'Guelmim', 'Kénitra', 'Laâyoune', 'Marrakech', 'Meknès', 'Oujda', 'Rabat', 'Safi', 'Settat', 'Tanger', 'Tétouan']


## Normalizing month casing

In [27]:
month_map = {
    "janvier":"Janvier","février":"Février","mars":"Mars","avril":"Avril","mai":"Mai","juin":"Juin",
    "juillet":"Juillet","août":"Août","septembre":"Septembre","octobre":"Octobre","novembre":"Novembre","décembre":"Décembre",
}

In [28]:
monthly_df["month"] = monthly_df["month"].str.lower().map(month_map)
yearly_df["month"] = yearly_df["month"].str.lower().map(month_map)
city_df["month"] = city_df["month"].str.lower().map(month_map)

print(monthly_df["month"].isna().sum(), yearly_df["month"].isna().sum(), city_df["month"].isna().sum())

0 0 0


In [29]:
print("monthly dup:", monthly_df.duplicated(subset=["category","month","year"], keep=False).sum())
print("yearly dup:", yearly_df.duplicated(subset=["category","month","year"], keep=False).sum())
print("city dup:", city_df.duplicated(subset=["city","month","year"], keep=False).sum())

print()
print("Final shapes:", monthly_df.shape, yearly_df.shape, city_df.shape)

monthly dup: 0
yearly dup: 0
city dup: 0

Final shapes: (2666, 6) (2595, 9) (3282, 6)


In [31]:
m_df= pd.read_csv("hcp_monthly_category_clean.csv")
m_df

,category,index_prev_period,index_curr_period,change_pct,month,year
0,Produits alimentaires,128.0,125.6,-1.9,Juillet,2026
1,01 - Produits alimentaires et boissons non alc...,127.2,124.6,-2.0,Juillet,2026
2,02 - Boissons alcoolisées et tabac,150.5,150.5,0.0,Juillet,2026
3,Produits non alimentaires,114.6,114.3,-0.3,Juillet,2026
4,03 - Articles d'habillement et chaussures,117.9,117.7,-0.2,Juillet,2026
...,...,...,...,...,...,...
2661,09 - Loisirs et culture,98.1,98.0,-0.1,Novembre,2009
2662,10 - Enseignement,109.1,113.3,3.8,Novembre,2009
2663,11 - Restaurants et hôtels,105.1,107.8,2.6,Novembre,2009
2664,12 - Biens et services divers,103.8,105.8,1.9,Novembre,2009


## Adding date column to all 3 datasets

In [32]:
month_to_num = {
    "Janvier": 1, "Février": 2, "Mars": 3, "Avril": 4, "Mai": 5, "Juin": 6,
    "Juillet": 7, "Août": 8, "Septembre": 9, "Octobre": 10, "Novembre": 11, "Décembre": 12,
}

In [33]:
monthly_df["date"] = pd.to_datetime(
    monthly_df["year"].astype(str) + "-" + monthly_df["month"].map(month_to_num).astype(str) + "-01"
)
yearly_df["date"] = pd.to_datetime(
    yearly_df["year"].astype(str) + "-" + yearly_df["month"].map(month_to_num).astype(str) + "-01"
)
city_df["date"] = pd.to_datetime(
    city_df["year"].astype(str) + "-" + city_df["month"].map(month_to_num).astype(str) + "-01"
)

In [35]:
monthly_df.head()

,category,index_prev_period,index_curr_period,change_pct,month,year,date
0,Produits alimentaires,128.0,125.6,-1.9,Juillet,2026,2026-07-01
1,01 - Produits alimentaires et boissons non alc...,127.2,124.6,-2.0,Juillet,2026,2026-07-01
2,02 - Boissons alcoolisées et tabac,150.5,150.5,0.0,Juillet,2026,2026-07-01
3,Produits non alimentaires,114.6,114.3,-0.3,Juillet,2026,2026-07-01
4,03 - Articles d'habillement et chaussures,117.9,117.7,-0.2,Juillet,2026,2026-07-01


In [39]:
# months available
coverage = monthly_df.groupby("year")["date"].nunique().reset_index()
coverage.columns = ["year", "months_available"]
coverage

,year,months_available
0,2009,2
1,2010,12
2,2011,6
3,2012,6
4,2013,12
5,2014,12
6,2015,12
7,2016,12
8,2017,12
9,2018,12


In [42]:
full_range = pd.date_range(start=monthly_df["date"].min(), end=monthly_df["date"].max(), freq="MS")
present_dates = set(monthly_df["date"].unique())
missing_dates = [d for d in full_range if d not in present_dates]

print(f"Expected months: {len(full_range)}")
print(f"Present months: {len(present_dates)}")
print(f"Missing months: {len(missing_dates)}")
for d in missing_dates:
    print(d.strftime("%Y-%m"))

Expected months: 201
Present months: 189
Missing months: 12
2011-07
2011-08
2011-09
2011-10
2011-11
2011-12
2012-01
2012-02
2012-03
2012-04
2012-05
2012-06


In [45]:
monthly_df.to_csv("/content/drive/MyDrive/morocco-purchasing-power/hcp_monthly_category_clean.csv", index=False)
yearly_df.to_csv("/content/drive/MyDrive/morocco-purchasing-power/hcp_yearly_category_clean.csv", index=False)
city_df.to_csv("/content/drive/MyDrive/morocco-purchasing-power/hcp_city_indices_clean.csv", index=False)
print("Saved.")

Saved.
